# Delivery Metrics and Monte Carlo Forecasting

Companion notebook for the [Delivery and Execution as a Manager](https://codemia.io/courses/staff_and_engineering_manager_interviews/stf_delivery_and_execution_as_a_manager) and [Multi Team Execution](https://codemia.io/courses/staff_and_engineering_manager_interviews/stf_multi_team_execution) lessons on Codemia.

Execution rounds ask how you plan, estimate and track work, and how you answer the question every leader hears: when will it be done? In this notebook you will:

1. Forecast a delivery date from past throughput with a Monte Carlo simulation and read it as a range.
2. Compare that with a single point estimate and see why single dates slip.
3. Add scope growth and watch the date move.
4. Use Little's law to see why limiting work in progress shortens cycle time.
5. Compute the four DORA delivery metrics from a deploy log.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## 1. Past throughput is the best estimate you have

Throughput is the number of work items a team finishes each week. It varies: holidays, incidents and big items make some weeks slow. Instead of averaging that away, a Monte Carlo forecast **resamples** real weeks to simulate many possible futures.

In [ ]:
history = np.array([4, 6, 3, 7, 5, 2, 6, 8, 5, 4, 0, 6, 7, 5, 3, 6, 9, 4, 5, 6, 2, 7, 5, 6, 4, 5])
print("weeks of history:", len(history), " mean per week:", history.mean().round(2))

plt.figure(figsize=(7, 3))
plt.bar(range(len(history)), history)
plt.xlabel("week"); plt.ylabel("items finished"); plt.title("Weekly throughput")
plt.show()

In [ ]:
def weeks_to_finish(backlog, history, trials=20_000, growth=None):
    out = np.empty(trials, dtype=int)
    for t in range(trials):
        left, w = float(backlog), 0
        while left > 0:
            left -= rng.choice(history)
            if growth is not None:
                left += rng.poisson(growth)
            w += 1
            if w > 500:
                break
        out[t] = w
    return out

backlog = 60
sims = weeks_to_finish(backlog, history)
p50, p85, p95 = np.percentile(sims, [50, 85, 95])
point = backlog / history.mean()
print(f"single point estimate: {point:.1f} weeks")
print(f"Monte Carlo: 50% by week {p50:.0f}, 85% by week {p85:.0f}, 95% by week {p95:.0f}")
assert p85 >= p50

In [ ]:
plt.figure(figsize=(7, 4))
vals, cnt = np.unique(sims, return_counts=True)
plt.bar(vals, cnt / cnt.sum())
for p, lab in [(p50, "50%"), (p85, "85%")]:
    plt.axvline(p, color="k", ls="--"); plt.text(p + .1, max(cnt / cnt.sum()) * .9, lab)
plt.axvline(point, color="r"); plt.text(point - 2.2, max(cnt / cnt.sum()) * .7, "point", color="r")
plt.xlabel("weeks to finish 60 items"); plt.ylabel("share of simulations")
plt.title("A forecast is a range, not a date")
plt.show()

The point estimate lands near the middle, which means it is missed about half the time. Committing to the **85th percentile** and sharing the range (the 50th percentile week as most likely, the 85th as very likely) is the kind of answer interviewers want when they ask how you set dates.

## 2. Scope grows while you work

Backlogs rarely stay fixed. Discovered work, bugs and stakeholder asks add items every week. Below, an average of 1.5 new items arrive each week.

In [ ]:
grown = weeks_to_finish(backlog, history, growth=1.5)
g50, g85 = np.percentile(grown, [50, 85])
print(f"fixed scope:   50% by week {p50:.0f}, 85% by week {p85:.0f}")
print(f"growing scope: 50% by week {g50:.0f}, 85% by week {g85:.0f}")
assert g85 > p85

Scope growth of about a third of throughput pushes the date out by weeks. That is the numeric case for the habits in the lesson: track scope added each week as its own line, and make explicit trade offs (cut, defer or move the date) when it grows, instead of hoping the team absorbs it.

## 3. Little's law: less work in progress, shorter cycle time

For a stable system, **average cycle time = average work in progress / throughput**. A team that starts everything at once does not finish sooner; each item just waits longer. The simulation below runs a team that can finish about five items a week with different limits on how many items are open.

In [ ]:
def run_team(wip_limit, weeks=200, capacity=5.0, arrivals=4.6):
    open_items, cycle, start_week, queue = [], [], {}, 0
    nid = 0
    for w in range(weeks):
        queue += rng.poisson(arrivals)
        while queue and len(open_items) < wip_limit:
            open_items.append(nid); start_week[nid] = w; nid += 1; queue -= 1
        done = min(len(open_items), rng.poisson(capacity))
        # with more items open, context switching spreads effort thin
        done = int(done * min(1.0, 6 / max(len(open_items), 1)) + 0.5)
        for _ in range(done):
            i = open_items.pop(0)
            cycle.append(w - start_week[i] + 1)
    return np.mean(cycle), len(cycle) / weeks

for limit in [3, 6, 12, 24]:
    ct, tp = run_team(limit)
    print(f"WIP limit {limit:2d}: cycle time {ct:5.1f} weeks, throughput {tp:4.1f} per week")

Past a point, raising the WIP limit makes items take longer and, with context switching, can even lower throughput. In an interview: "We cut work in progress from twelve to six and cycle time halved, without hurting throughput" is a concrete execution story with a mechanism behind it.

## 4. The four DORA metrics from a deploy log

The DORA metrics are a common shared language for delivery health: **deployment frequency**, **lead time for changes**, **change failure rate** and **time to restore service**. Below we generate a quarter of deploys and compute each.

In [ ]:
n = 90
deploy_day = np.sort(rng.uniform(0, 90, n))
lead_hours = rng.lognormal(mean=np.log(30), sigma=0.7, size=n)      # commit to production
failed = rng.random(n) < 0.12
restore_hours = np.where(failed, rng.lognormal(np.log(2.5), 0.8, n), np.nan)

freq = n / (90 / 7)
lead = np.median(lead_hours)
cfr = failed.mean()
mttr = np.nanmedian(restore_hours)
print(f"deployment frequency: {freq:.1f} per week")
print(f"lead time for changes (median): {lead:.0f} hours")
print(f"change failure rate: {cfr:.0%}")
print(f"time to restore (median): {mttr:.1f} hours")

In [ ]:
weeks = (deploy_day // 7).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(*np.unique(weeks, return_counts=True)); ax[0].set_title("deploys per week"); ax[0].set_xlabel("week")
ax[1].hist(lead_hours, bins=30); ax[1].set_title("lead time (hours)")
plt.tight_layout(); plt.show()

Use these as **trends for the team**, not targets for individuals: a team that is pushed on deploy count alone will split changes artificially, and one pushed on change failure rate alone will stop shipping. Pair each with its guardrail, as the business judgment lesson describes.

## Exercises

1. Replace `history` with twelve weeks from a real team (or your own estimates). How wide is the gap between the 50th and 85th percentiles?
2. Forecast the other way: how many items will be done **by week 8**? Simulate eight weeks of throughput many times and report the 85th percentile *lowest* count, which is what you can safely promise.
3. Split the 60 items across two teams that must both finish. Forecast each and take the later finish per simulation. Why is the joint date later than either team's own forecast?